# 12 — Agentic RAG

Module 11 retrieved once and generated. That is enough when one file holds the answer.

Real questions often do not. *"The same title arrived damaged twice — what do we offer the customer, and how long does the refund take after approval?"* requires two distinct facts:
1. *What is our policy when an item arrives damaged a second time?* (Offer a refund instead of a third copy).
2. *How long does a refund take after approval?* (5 to 10 business days).

Those facts live in two separate files (`policy_damaged_media.md` and `policy_refunds.md`). A single nearest-neighbour search at `k=1` will only grab one of them.

Today retrieval becomes a **tool**. The model inspects what it retrieved, determines what facts are still missing, and decides whether to search again or stop. That is agentic RAG. The store did not change. Chroma is still Chroma.

**Module 11 — one retrieve, one generate**

```mermaid
flowchart LR
    A["question"] --> B["retrieve k=3"]
    B --> C["generate"]
```

**Module 12 — retrieve is a tool, and the model drives the search**

```mermaid
graph TD
    A[Question] --> B[Model requests search]
    B --> C[Execute retrieve tool]
    C --> D[Feed document back to model]
    D --> E{All facts found?}
    E -->|Yes| F[Generate final answer]
    E -->|Missing a fact| B
    E -->|Not in corpus| G[Say not found]
    G --> H[END]
```

### What's the same as 11, what's different

| | Module 11 | Module 12 |
|---|---|---|
| Corpus | Same 36 files | Same 36 files |
| Store | Chroma, in memory | Chroma, in memory |
| The embedding call | Same shape | Same shape |
| How many times we search | Exactly once, every time | The model decides — once, twice, or gives up |
| Who decides to search again | Nobody. Your code retrieves once. | The model. `retrieve` becomes a **tool**, the same idea as `get_fact` in module 02. |

If none of that is unfamiliar, this whole module is one core idea: hand the model a `retrieve` tool, inspect the missing facts after each turn, and let the model hop across documents until it has the full picture.

## 1. Learn

```
11  one retrieve, one generate
12  you are here — retrieve is a tool; the model hops across documents
14  a retrieved file can carry instructions
```

Single-shot RAG fails on at least three distinct patterns:

| Shape | Why one retrieve is not enough |
|---|---|
| Multi-hop | The second fact is not in the first file. |
| Bad phrasing | The user said "SLA"; the file said "first reply." |
| Absent | Nearest is still a file. The generate step has to refuse. |

The agentic move is not a new vector database. It is tool calling applied to retrieval:

| Step | What happens |
|---|---|
| Formulate | Turn the user's question into focused search queries. |
| Retrieve | Search Chroma for the nearest document. |
| Assess | Does what I found actually answer the entire question? |
| Re-retrieve | If a fact is missing, search specifically for the missing topic. |
| Synthesize | Combine all retrieved facts into a single accurate response. |

```
Turn 1: retrieve("damaged twice policy")
           observation: policy_damaged_media.md  (refund, no third copy)

Turn 2: retrieve("refund processing timeline")
           observation: policy_refunds.md  (5 to 10 business days)

Turn 3: stop — both facts are in the context window
```

Always cap the maximum turns. An unanswerable question would otherwise search forever.

## 2. Do

### Step 1: Same folder, same embeddings, in this kernel

No imports from module 11. We index the 36 markdown files into Chroma so this notebook stands completely alone.

In [1]:
from pathlib import Path
import json
import os

from chromadb import Client
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI


load_dotenv(find_dotenv(usecwd=True))
ROOT = Path(find_dotenv(usecwd=True)).parent

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
model = os.environ.get("MODEL_DEFAULT", "").strip()
embed_model = os.environ.get("EMBEDDING_MODEL", "").strip()
assert api_key, "OPENAI_API_KEY is missing."
assert model, "MODEL_DEFAULT is missing from .env."
assert embed_model, "EMBEDDING_MODEL is missing. Copy the line from .env.example."

client = OpenAI()
CORPUS = ROOT / "data" / "corpus"
files = sorted(CORPUS.glob("*.md"))


def embed(text: str) -> list[float]:
    return client.embeddings.create(model=embed_model, input=text).data[0].embedding


ids, documents, metadatas, vectors = [], [], [], []
for path in files:
    text = path.read_text()
    ids.append(path.name)
    documents.append(text)
    metadatas.append({"path": path.name})
    vectors.append(embed(text))

chroma = Client()
collection = chroma.get_or_create_collection("corpus")
collection.add(ids=ids, documents=documents, metadatas=metadatas, embeddings=vectors)
print("OPENAI_API_KEY is set:", True)
print("MODEL_DEFAULT:", model)
print("stored:", collection.count())

OPENAI_API_KEY is set: True
MODEL_DEFAULT: gpt-5.4-nano
stored: 36


### Step 2: `retrieve` is an ordinary function

Print `k=2` once so you can see that **both** files exist in our corpus. The tool we give the model will enforce `k=1`, meaning a single retrieve call cannot carry both facts.

In [2]:
def retrieve(query: str, k: int = 2):
    got = collection.query(query_embeddings=[embed(query)], n_results=k)
    rows = []
    for i in range(len(got["ids"][0])):
        rows.append(
            {
                "id": got["ids"][0][i],
                "distance": got["distances"][0][i],
                "document": got["documents"][0][i],
            }
        )
    return rows


QUESTION = (
    "If the same title arrives damaged twice, "
    "what do we offer the customer, and how long does the refund take after approval?"
)
for row in retrieve(QUESTION, k=2):
    print(f"{row['distance']:.3f}  {row['id']}")
    print(row["document"].splitlines()[0])
    print()

0.737  policy_damaged_media.md
# Damaged media

0.915  policy_refunds.md
# Refunds



Two files: `policy_damaged_media.md` contains the second-time rule (*offer refund, no third copy*), while `policy_refunds.md` contains the timeline (*5 to 10 business days*).

### Step 3: Single-shot RAG (`k=1`) — The Failure Case

With single-shot retrieval at `k=1`, the model only receives `policy_damaged_media.md`. It has no way of knowing how long a refund takes.

In [5]:
def generate(question: str, rows):
    packed = "\n\n".join(f"# {row['id']}\n{row['document']}" for row in rows)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer using only the documents. "
                    "If they do not contain the answer, say you do not know. "
                    "Do not invent a number, a date, or a phone number."
                ),
            },
            {
                "role": "user",
                "content": "Documents:\n\n" + packed + "\n\nQuestion: " + question,
            },
        ],
        max_completion_tokens=160,
        reasoning_effort="none",
    )
    return response.choices[0].message.content, response.usage.prompt_tokens


one = retrieve(QUESTION, k=1)
text_one, tokens_one = generate(QUESTION, one)
print("used:", [row["id"] for row in one])
print("prompt_tokens:", tokens_one)
print()
print(text_one)

used: ['policy_damaged_media.md']
prompt_tokens: 141

If the same title arrives damaged twice, we offer a **refund instead of a third copy**.  

However, the documents provided do **not** state **how long the refund takes after approval**, so I do not know.


Single-shot retrieval failed to answer the full question because it only saw one document.

Now let's build agentic retrieval. Rather than jumping straight into an automated loop, we will step through the turns by hand so you can see the model request search queries, receive document payloads, and synthesize the full answer.

### Step 4: Define the `retrieve` tool and schema

In [7]:
def schema(tool_name, description, **props):
    return {
        "type": "function",
        "function": {
            "name": tool_name,
            "description": description,
            "parameters": {"type": "object", "properties": props, "required": list(props)},
        },
    }


tools = [
    schema(
        "retrieve",
        "Search the shop policy and customer ticket files. "
        "Argument query must be a concise keyword phrase (e.g. 'damaged items', 'refund timeline', 'Helena ticket'). "
        "Do not repeat keywords from files already retrieved.",
        query={"type": "string"},
    ),
]

# tool schema
tools

[{'type': 'function',
  'function': {'name': 'retrieve',
   'description': "Search the shop policy and customer ticket files. Argument query must be a concise keyword phrase (e.g. 'damaged items', 'refund timeline', 'Helena ticket'). Do not repeat keywords from files already retrieved.",
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string'}},
    'required': ['query']}}}]

In [8]:
def run_retrieve(call):
    args = json.loads(call.function.arguments or "{}")
    query = args.get("query", "")
    rows = retrieve(query, k=1)
    names = [row["id"] for row in rows]
    packed = []
    for row in rows:
        packed.append(f"{row['id']} (distance {row['distance']:.3f})\n{row['document']}")
    return query, names, "\n\n".join(packed)


SYSTEM = (
    "You answer questions using only the shop policy and ticket files. Use the retrieve tool to find facts.\n"
    "Each search query should be 2 to 4 keywords for one specific concept (e.g. 'damaged items' or 'refund timeline').\n"
    "If you already retrieved a document, do not repeat its terms in subsequent search queries.\n"
    "If the files do not contain the answer, say you do not know.\n"
    "Do not invent facts."
)

print([t["function"]["name"] for t in tools])

['retrieve']


### Step 5: Turn 1 (Manual) — The model asks for documents

We start the conversation by sending the question. Notice the model does not guess; it emits `tool_calls` for the `retrieve` function.

In [9]:
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": QUESTION},
]

response_1 = client.chat.completions.create(
    model=model,
    messages=messages,
    tools=tools,
    max_completion_tokens=200,
    reasoning_effort="none",
)
msg_1 = response_1.choices[0].message
print("Finish reason:", response_1.choices[0].finish_reason)
print("Tool calls count:", len(msg_1.tool_calls or []))
print()

messages.append(msg_1)
for call in msg_1.tool_calls or []:
    query_1, names_1, content_1 = run_retrieve(call)
    print("Query:", query_1)
    print("File: ", names_1)
    print()
    messages.append({"role": "tool", "tool_call_id": call.id, "content": content_1})

Finish reason: tool_calls
Tool calls count: 3

Query: damaged items policy
File:  ['policy_damaged_media.md']

Query: refund processing time approval
File:  ['policy_refunds.md']

Query: repeat damaged delivery
File:  ['policy_damaged_media.md']



### Step 6: Turn 2 (Manual) — The "Aha!" Moment and Synthesis

Now we call the model again with the updated message history containing the retrieved files.

The model reads the retrieved documents. Seeing that both facts are present (`policy_damaged_media.md` for the damaged replacement rule, and `policy_refunds.md` for the 5-10 day refund timeline), `finish_reason` becomes `stop` and the model produces the complete, grounded answer!

In [10]:
response_2 = client.chat.completions.create(
    model=model,
    messages=messages,
    tools=tools,
    max_completion_tokens=200,
    reasoning_effort="none",
)
msg_2 = response_2.choices[0].message
print("Finish reason:", response_2.choices[0].finish_reason)
print()
print(msg_2.content)

Finish reason: stop

If the same title arrives damaged twice, we offer the customer a **refund instead of a third replacement copy**.

After support approves the refund request, it takes **5 to 10 business days** to return to the original payment method.


### Step 7: Packaging into `run_loop`

Now that you have seen the mechanism step-by-step:
1. Send `messages` with `tools=tools`.
2. If `tool_calls` present: execute `retrieve()`, append results, and repeat.
3. If no `tool_calls`: return the final content.

We can package this exact pattern into a clean loop function with a safety turn cap of 5.

In [11]:
def run_loop(question, max_turns=5):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    queries = []
    files_used = []
    final_text = None
    prompt_tokens = 0
    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            max_completion_tokens=200,
            reasoning_effort="none",
        )
        prompt_tokens = prompt_tokens + response.usage.prompt_tokens
        message = response.choices[0].message
        print("--- turn", turn + 1, "finish_reason:", response.choices[0].finish_reason, "---")
        if not message.tool_calls:
            final_text = message.content
            print(final_text)
            break
        messages.append(message)
        for call in message.tool_calls:
            query, names, result = run_retrieve(call)
            queries.append(query)
            files_used.extend(names)
            print("retrieve:", query)
            print("  files:", names)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
    else:
        print("stopped: cap was", max_turns)
    return {
        "queries": queries,
        "files_used": files_used,
        "n_retrieves": len(queries),
        "final_text": final_text,
        "prompt_tokens": prompt_tokens,
    }


looped = run_loop(QUESTION)
print()
print("n_retrieves:", looped["n_retrieves"])
print("queries:    ", looped["queries"])

--- turn 1 finish_reason: tool_calls ---
retrieve: damaged items twice
  files: ['policy_damaged_media.md']
retrieve: refund after approval timeline
  files: ['policy_refunds.md']
retrieve: damaged title replacement refund
  files: ['policy_damaged_media.md']
--- turn 2 finish_reason: stop ---
If the same title arrives damaged **twice**, we **offer a refund instead of a third replacement copy**.

After support **approves** the refund request, it takes **5 to 10 business days** to be returned to the original payment method.

n_retrieves: 3
queries:     ['damaged items twice', 'refund after approval timeline', 'damaged title replacement refund']


### Step 8: An unanswerable question

The corpus has no CEO personal phone number. The loop should search once, see that the retrieved escalation document does not contain personal phone numbers, and stop honestly without looping endlessly.

In [12]:
absent = run_loop("What is the CEO's personal mobile number?")
print()
print("n_retrieves:", absent["n_retrieves"])
print("queries:    ", absent["queries"])

--- turn 1 finish_reason: tool_calls ---
retrieve: CEO mobile number
  files: ['policy_escalations.md']
--- turn 2 finish_reason: stop ---
I don’t know. The shop policy files state that the shop does **not** publish the manager’s personal phone number, and I don’t have any source for the CEO’s personal mobile number.

n_retrieves: 1
queries:     ['CEO mobile number']


## 3. Observe

Same corpus, two retrieval patterns:

1. **Single-shot RAG** is cheap and fast, but fails whenever an answer spans multiple documents or requires query reformulation.
2. **Agentic RAG** can answer multi-hop questions and handle missing facts, but consumes more round-trips and tokens per question.

In [13]:
print(f"{'method':<16} {'n_retrieves':>11}  {'top / files'}")
print(f"{'single-shot k=1':<16} {1:11}  {[row['id'] for row in one]}")
print(f"{'loop, damaged':<16} {looped['n_retrieves']:11}  {looped['files_used']}")
print(f"{'loop, CEO':<16} {absent['n_retrieves']:11}  {absent['files_used']}")
print()
print("single-shot tokens:", tokens_one)
print("loop tokens:       ", looped["prompt_tokens"])
print("CEO loop tokens:   ", absent["prompt_tokens"])

method           n_retrieves  top / files
single-shot k=1            1  ['policy_damaged_media.md']
loop, damaged              3  ['policy_damaged_media.md', 'policy_refunds.md', 'policy_damaged_media.md']
loop, CEO                  1  ['policy_escalations.md']

single-shot tokens: 141
loop tokens:        901
CEO loop tokens:    631


Things to notice:

- `retrieve`'s `query` is a **string the model wrote**. You can log it, inspect it, or guard against malicious queries.
- Multi-hop retrieval costs more tokens. Do not use an agent loop when single-shot retrieval is sufficient.
- The CEO question still returned nearest files (`policy_escalations.md`). Nearest does not mean relevant. The prompt constraint is what forced the model to refuse rather than invent.

## 4. Challenge

Use `run_loop` to answer a new multi-hop question:

> Helena received the wrong album. Who is her assigned support representative, and how soon should support send a first reply?

The customer ticket (`ticket_01_helena_wrong_album.md`) names the representative (**Steve Johnson**). The first-reply SLA is in the support-hours policy (`policy_support_hours.md`). A single retrieval cannot answer both.

Bind:
- `n_retrieves` — how many times `retrieve` was called
- `files_used` — the list of filenames returned
- `final_text` — the final response string

The assertion cell verifies that you retrieved at least twice, found Steve Johnson, and included the 1-business-day reply timeframe.

In [ ]:
# n_retrieves, files_used, final_text = ...

In [ ]:
assert n_retrieves >= 2, "this question needs a second retrieve"
assert files_used, "files_used should be the names retrieve returned"
assert final_text and str(final_text).strip(), "final_text should be the last sentence"
assert "steve" in str(final_text).lower(), "Helena's assigned rep is Steve Johnson"
assert "business" in str(final_text).lower(), "first reply is within 1 business day"
print("looks good")